# Model deployment

## Import usual packages

In [6]:
import torch
import torchvision
import torchinfo as ti

In [20]:
import torch.nn as nn

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [10]:
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights, vit_b_16, ViT_B_16_Weights

In [23]:
from pathlib import Path
import os

In [24]:
from zipfile import ZipFile

## Constant

In [12]:
SIZE_INFO = (1,3,224,224)

## Creating an EfficientNet B2 model

In [7]:
eff_net_model = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)

In [13]:
ti.summary(eff_net_model, input_size=SIZE_INFO)

Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [1, 1000]                 --
├─Sequential: 1-1                                       [1, 1408, 7, 7]           --
│    └─Conv2dNormActivation: 2-1                        [1, 32, 112, 112]         --
│    │    └─Conv2d: 3-1                                 [1, 32, 112, 112]         864
│    │    └─BatchNorm2d: 3-2                            [1, 32, 112, 112]         64
│    │    └─SiLU: 3-3                                   [1, 32, 112, 112]         --
│    └─Sequential: 2-2                                  [1, 16, 112, 112]         --
│    │    └─MBConv: 3-4                                 [1, 16, 112, 112]         1,448
│    │    └─MBConv: 3-5                                 [1, 16, 112, 112]         612
│    └─Sequential: 2-3                                  [1, 24, 56, 56]           --
│    │    └─MBConv: 3-6                                

## Creating a ViT-Base model

In [11]:
vit_model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

In [14]:
ti.summary(vit_model, input_size=SIZE_INFO)

Layer (type:depth-idx)                        Output Shape              Param #
VisionTransformer                             [1, 1000]                 768
├─Conv2d: 1-1                                 [1, 768, 14, 14]          590,592
├─Encoder: 1-2                                [1, 197, 768]             151,296
│    └─Dropout: 2-1                           [1, 197, 768]             --
│    └─Sequential: 2-2                        [1, 197, 768]             --
│    │    └─EncoderBlock: 3-1                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-2                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-3                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-4                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-5                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-6                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-7             

## Create the classificatgion head

In [18]:
eff_net_model.classifier

Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1408, out_features=1000, bias=True)
)

In [19]:
vit_model.heads

Sequential(
  (head): Linear(in_features=768, out_features=1000, bias=True)
)

In [21]:
def create_classification_head(model:str):
    if model.lower() != "vit":
        in_feature = 1408
        class_head = nn.Sequential(
            Dropout(p=0.3, inplace=True),
            Linear(in_features=in_feature, out_features=3, bias=True)
        )
        return class_head
    return nn.Sequential(Linear(in_features=768, out_features=3, bias=True))

## Downloading the data

In [22]:
!wget https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip

--2026-07-14 17:12:34--  https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/data/pizza_steak_sushi_20_percent.zip [following]
--2026-07-14 17:12:35--  https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/data/pizza_steak_sushi_20_percent.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8001::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8001::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31491084 (30M) [application/zip]
Saving to: ‘pizza_steak_sushi_20_percent.zip’

pizza_steak_sushi_2 100%[===================>]  30.03M  1.34MB/s  

In [32]:
current_folder = Path(".")
for file in current_folder.iterdir():
    if file.suffix == ".zip":
        with  ZipFile("pizza_steak_sushi_20_percent.zip", mode="r") as file:
            file.extractall("data/")
        os.remove("pizza_steak_sushi_20_percent.zip")
        print("Zip file deleted and extracted")

Zip file deleted and extracted


In [33]:
train_dir = Path(".")/ "data"/ "train"
test_dir = Path(".") / "data" / "test"

In [34]:
# Check if the folders are well located in the train and test directory
for folder in train_dir.iterdir():
    print(folder)

data/train/steak
data/train/sushi
data/train/pizza


## Creating the training and testing loop